# CommGuard: Kaggle dual-T4 validation

**Independent, unofficial research prototype. Not affiliated with or endorsed by SPAR, Kairos, ERA, UChicago XLab, William Fowler, or cited authors/institutions.**

This notebook is limited to one host with two NVIDIA T4 GPUs. NVML PCIe TX/RX values are PCIe traffic readings, not complete NCCL or GPU-to-GPU byte measurements. No frontier-scale, privacy, production, security, or treaty claim follows from this study.

In [ ]:
from pathlib import Path
import subprocess, sys

REPO_URL = 'https://github.com/waqasm86/CommGuard.git'
GIT_REF = 'main'  # Replace with a reviewed commit SHA for a reproducible run.
REPO = Path('/kaggle/working/CommGuard')
if not (REPO / 'pyproject.toml').is_file():
    subprocess.run(['git', 'clone', '--filter=blob:none', '--no-checkout', REPO_URL, str(REPO)], check=True)
    subprocess.run(['git', '-C', str(REPO), 'fetch', '--depth', '1', 'origin', GIT_REF], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
ARTIFACTS = Path('/kaggle/working/commguard-artifacts')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-build-isolation', '--no-deps', '-e', str(REPO)], check=True)
print('Repository:', REPO)
print('Artifacts:', ARTIFACTS)

## Strict environment and topology gate

In [ ]:
from commguard.environment.preflight import check_environment, summarize_environment

environment = check_environment(strict=True, output=ARTIFACTS)
print(summarize_environment(environment))
print(environment['topology']['stdout'])

## Smallest two-rank NCCL smoke test

The worker subprocess must emit distinct rank-to-GPU bindings, a CUDA operation, heartbeat, completion, and cleanup evidence for both ranks.

In [ ]:
from commguard.orchestrator import run_experiment

smoke = run_experiment(
    'collective_all_reduce_1mib',
    output=ARTIFACTS,
    overrides={'iterations': 2},
    timeout_s=120,
)
print(smoke['run_id'], smoke['manifest']['participation_valid'])
assert smoke['manifest']['participation_valid']

## Communication calibration gate

Nominal tensor payload, PyTorch/NCCL timing, and NVML readings remain separate evidence channels. A negative result stops the main corpus by default.

In [ ]:
from commguard.orchestrator import run_calibration_sweep

calibration = run_calibration_sweep(
    output=ARTIFACTS,
    payload_mib=(1, 4, 16, 64),
    repetitions=1,
    timeout_s=180,
)
print('Calibration:', calibration['status'])
print('Reasons:', calibration['falsification_reasons'])
CALIBRATION_SUPPORTED = calibration['status'] == 'supported'

## Bounded experiment profile

Expensive collection is opt-in. The standard profile uses three whole-run repetitions and retains OOMs/failures without hidden batch-size changes.

In [ ]:
from commguard.orchestrator import estimate_matrix, run_matrix

print(estimate_matrix('standard', repetitions=3))
RUN_STANDARD = False
if RUN_STANDARD:
    if not CALIBRATION_SUPPORTED:
        raise RuntimeError('Calibration did not support detector collection in this session.')
    matrix = run_matrix('standard', output=ARTIFACTS, repetitions=3, timeout_s=240)
    print(matrix)
else:
    print('Standard profile skipped; set RUN_STANDARD=True explicitly.')

## Feature derivation, grouped evaluation, and report

In [ ]:
from commguard.features import extract_features
from commguard.evaluation import evaluate_detector
from commguard.reporting import generate_report

rows = extract_features(ARTIFACTS, output=ARTIFACTS, window_lengths=(5, 15, 30))
print('Feature rows:', len(rows))
if RUN_STANDARD and rows:
    results = evaluate_detector(ARTIFACTS, output=ARTIFACTS)
    print(results['ablations']['combined'])
report_path = ARTIFACTS / 'report.md'
print(generate_report(ARTIFACTS, output=report_path)[:2000])

## Export complete evidence

In [ ]:
from commguard.artifacts import ArtifactStore

archive = ArtifactStore(ARTIFACTS).export('/kaggle/working/commguard-artifacts.tar.gz')
print(archive)